# Ride Demand Forecasting
### Hourly Uber/Lyft Demand in Boston - Temporal Aggregation & MLForecast

**Project 24 of 50 - Time Series Forecasting Portfolio**

## Project Overview

Ride-hailing demand varies by hour, day-of-week, weather, and surge dynamics. This dataset captures individual Uber and Lyft trips in Boston, MA in late 2018. We aggregate them to **hourly trip counts** and forecast the next 24 hours.

| Attribute | Value |
|---|---|
| **Dataset** | `ravi72munde/uber-lyft-cab-prices` |
| **Target** | Hourly ride count (aggregated from individual trips) |
| **Date column** | `timestamp` (Unix epoch) - convert to datetime |
| **Frequency** | Hourly (after aggregation) |
| **TS Library** | MLForecast |
| **Covariates** | temperature, rain, humidity, wind, clouds |

## Learning Objectives

1. **Aggregate transaction-level data** into a time series: ride counts per hour
2. Understand **surge pricing dynamics** - does high demand cause surge, or does surge suppress demand?
3. Compare Uber vs. Lyft demand patterns for the same city
4. Build MLForecast with weather covariates for a ride-hailing demand problem
5. Apply **multiplicative seasonal decomposition** to understand the intraday demand curve

## Problem Statement

Given 2 months of hourly Uber/Lyft demand in Boston (Nov-Dec 2018), forecast the next **24 hours** of total ride demand.

This horizon is used by ride-hailing platforms to pre-position driver supply before demand spikes.

## Why This Matters

- Uber and Lyft dispatch systems forecast demand 15-60 min ahead to pre-route nearby drivers and reduce ETAs
- Platform operators use multi-day forecasts to incentivise drivers to work during predicted busy periods
- City transportation agencies use demand forecasts to adjust public transit frequency (complementary or competing with ride-hailing)
- Surge pricing algorithms need demand forecasts to decide when to activate surge multipliers proactively
- Airport operators use ride-hailing demand forecasts to size pickup/dropoff kerb space

## Dataset Overview

**Source:** Kaggle - `ravi72munde/uber-lyft-cab-prices`

**Raw structure (individual trip rows):**
| Column | Description |
|---|---|
| `id` | Unique trip ID |
| `timestamp` | Unix epoch for ride request time |
| `cab_type` | Uber or Lyft |
| `name` | Service type: UberX, UberPool, Lyft, Lyft XL, etc. |
| `price` | Fare in USD (with surge already applied) |
| `distance` | Trip distance (miles) |
| `surge_multiplier` | Surge multiplier at time of booking |
| `source` / `destination` | Boston neighbourhoods |
| `temp` | Temperature (Fahrenheit) |
| `rain` | Precipitation (inches) |
| `humidity` | Relative humidity |
| `wind` | Wind speed (mph) |
| `clouds` | Cloud cover (%) |

**Period:** Nov-Dec 2018 | **Coverage:** Boston, MA

**We aggregate:** count rides per hour -> target time series

## Dataset Source & License

- **Kaggle slug:** `ravi72munde/uber-lyft-cab-prices`
- **License:** CC0 Public Domain
- **Primary use:** Originally published for price forecasting; repurposed here for demand volume forecasting

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ["kagglehub","pandas","numpy","matplotlib","seaborn","plotly",
            "scikit-learn","lazypredict","flaml","mlforecast","lightgbm","xgboost"]:
    try: __import__(pkg.split("[")[0].replace("-","_"))
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")

## Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from mlforecast import MLForecast
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import Ridge

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
print("Imports OK")

## Configuration & Constants

In [ ]:
PROJECT      = "Ride Demand Forecasting"
KAGGLE_SLUG  = "ravi72munde/uber-lyft-cab-prices"
TARGET       = "ride_count"     # after aggregation
SEASON_H     = 24    # daily seasonality
SEASON_W     = 168   # weekly
HORIZON      = 24    # 24h forecast
TEST_SIZE    = HORIZON * 7    # 1 week test
VAL_SIZE     = HORIZON * 7
RANDOM_STATE = 42
FLAML_BUDGET = 120

DATA_DIR = pathlib.Path("data"); DATA_DIR.mkdir(exist_ok=True)
print(f"Horizon: {HORIZON}h | Season: {SEASON_H}h | Test window: {TEST_SIZE}h = {TEST_SIZE//24} days")

## Kaggle Credential Check

In [ ]:
cred = (os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or
        os.environ.get("KAGGLE_API_TOKEN") or
        (pathlib.Path.home()/".kaggle"/"kaggle.json").exists())
if cred: print("Kaggle credentials found.")
else: print("WARNING: Set KAGGLE_USERNAME + KAGGLE_KEY env vars or ~/.kaggle/kaggle.json")

## Dataset Download & Loading

In [ ]:
import kagglehub
try:
    data_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
    print(f"Data at: {data_path}")
except Exception as e:
    print(f"kagglehub failed: {e}")
    os.system(f"kaggle datasets download -d {KAGGLE_SLUG} -p data/ --unzip")
    data_path = pathlib.Path("data")

csv_files = list(data_path.rglob("*.csv"))
print("Files:"); [print(f"  {f.name}  {f.stat().st_size/1e3:.0f}KB") for f in csv_files]

In [ ]:
if not csv_files:
    raise FileNotFoundError("No CSV found. Check credentials and slug.")

main_file = next((f for f in csv_files if "cab" in f.name.lower() or "rideshare" in f.name.lower()),
                  csv_files[0])
print(f"Loading: {main_file.name}")
df_raw = pd.read_csv(main_file)
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(3)

## Aggregate to Hourly Ride Demand

In [ ]:
# Convert unix timestamp to datetime
if "timestamp" in df_raw.columns:
    df_raw["datetime"] = pd.to_datetime(df_raw["timestamp"], unit="s", errors="coerce")
elif "date_time" in df_raw.columns:
    df_raw["datetime"] = pd.to_datetime(df_raw["date_time"], errors="coerce")
else:
    date_col = [c for c in df_raw.columns if "time" in c.lower() or "date" in c.lower()][0]
    df_raw["datetime"] = pd.to_datetime(df_raw[date_col], errors="coerce")

df_raw["hour_bucket"] = df_raw["datetime"].dt.floor("h")
print(f"Datetime range: {df_raw['datetime'].min()} -> {df_raw['datetime'].max()}")
print(f"Total trips   : {len(df_raw):,}")

In [ ]:
# Count rides per hour - total and by cab type
hourly = df_raw.groupby("hour_bucket").size().reset_index(name="ride_count")
print(f"Hourly series: {len(hourly)} observations")
print(hourly["ride_count"].describe().round(1))

# Weather covariates: median per hour bucket
weather_cols = [c for c in ["temp","rain","humidity","wind","clouds"] if c in df_raw.columns]
if weather_cols:
    weather_hourly = df_raw.groupby("hour_bucket")[weather_cols].median()
    hourly = hourly.merge(weather_hourly, on="hour_bucket", how="left")
    print(f"Weather covariates merged: {weather_cols}")

# Surge multiplier
if "surge_multiplier" in df_raw.columns:
    surge_hourly = df_raw.groupby("hour_bucket")["surge_multiplier"].mean()
    hourly = hourly.merge(surge_hourly.rename("avg_surge"), on="hour_bucket", how="left")
    print(f"Avg surge multiplier range: {hourly['avg_surge'].min():.2f} - {hourly['avg_surge'].max():.2f}")

hourly = hourly.rename(columns={"hour_bucket":"ds"}).sort_values("ds").reset_index(drop=True)
ts_df  = hourly.rename(columns={"ride_count":"y"})
print(ts_df.head(3))

## Data Validation Checks

In [ ]:
print("DATA VALIDATION REPORT")
print("="*55)
miss = ts_df["y"].isna().sum()
print(f"Target NaN   : {miss}")
print(f"Date range   : {ts_df['ds'].min()} -> {ts_df['ds'].max()}")
print(f"Hourly obs   : {len(ts_df)}")
print(f"Ride range   : {ts_df['y'].min()} - {ts_df['y'].max()} rides/hour")
print(f"Zero hours   : {(ts_df['y']==0).sum()} (expected ~0 at 3-5AM)")
for col in weather_cols + (["avg_surge"] if "avg_surge" in ts_df.columns else []):
    if col in ts_df.columns:
        print(f"{col}: {ts_df[col].min():.2f} - {ts_df[col].max():.2f}")

## Exploratory Data Analysis

In [ ]:
fig = px.line(ts_df, x="ds", y="y",
              title="Hourly Ride Demand in Boston (Uber + Lyft)",
              labels={"y":"Rides per Hour","ds":"Date"})
fig.update_layout(template="plotly_white"); fig.show()

In [ ]:
# Hourly and daily profiles
ts_df["hour"] = ts_df["ds"].dt.hour
ts_df["dow"]  = ts_df["ds"].dt.dayofweek

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
wd  = ts_df[ts_df["dow"] < 5].groupby("hour")["y"].mean()
wkd = ts_df[ts_df["dow"] >= 5].groupby("hour")["y"].mean()
axes[0].plot(wd.index, wd.values, label="Weekday", color="steelblue", lw=2)
axes[0].plot(wkd.index, wkd.values, label="Weekend", color="coral", lw=2)
axes[0].set_title("Hourly Ride Demand - Weekday vs Weekend")
axes[0].set_xlabel("Hour of Day"); axes[0].set_ylabel("Mean Rides/Hour")
axes[0].legend()

dow_avg = ts_df.groupby("dow")["y"].mean()
axes[1].bar(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"], dow_avg.values, color="teal", alpha=0.8)
axes[1].set_title("Average Demand by Day of Week")
plt.tight_layout(); plt.show()

In [ ]:
# Weather correlation with demand
if "temp" in ts_df.columns:
    fig = px.scatter(ts_df, x="temp", y="y", trendline="lowess",
                     title="Temperature vs Ride Demand",
                     labels={"temp":"Temperature (F)","y":"Rides/Hour"})
    fig.update_layout(template="plotly_white"); fig.show()
    print(f"Temp vs demand correlation: {ts_df['temp'].corr(ts_df['y']):.4f}")

if "avg_surge" in ts_df.columns:
    print(f"Surge vs demand correlation: {ts_df['avg_surge'].corr(ts_df['y']):.4f}")
    print("(Positive = surge occurs when demand is high, as expected)")

In [ ]:
# ACF
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(ts_df["y"].dropna(), lags=200, ax=axes[0], title="ACF (200 lags)")
plot_pacf(ts_df["y"].dropna(), lags=50, ax=axes[1], title="PACF (50 lags)")
plt.tight_layout(); plt.show()

## Uber vs Lyft Comparison (if data available)

In [ ]:
if "cab_type" in df_raw.columns:
    for cab in df_raw["cab_type"].unique():
        sub = df_raw[df_raw["cab_type"]==cab].groupby("hour_bucket").size()
        print(f"{cab}: {len(sub)} hours, mean {sub.mean():.0f} rides/hr, max {sub.max()}")
    
    uber_h = df_raw[df_raw["cab_type"]=="Uber"].groupby("hour_bucket").size().reset_index(name="uber")
    lyft_h = df_raw[df_raw["cab_type"]=="Lyft"].groupby("hour_bucket").size().reset_index(name="lyft")
    comp = uber_h.merge(lyft_h, on="hour_bucket").head(24*7)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=comp["hour_bucket"], y=comp["uber"], name="Uber"))
    fig.add_trace(go.Scatter(x=comp["hour_bucket"], y=comp["lyft"], name="Lyft"))
    fig.update_layout(title="Uber vs Lyft Hourly Demand (1 week)", template="plotly_white"); fig.show()
else:
    print("cab_type column not found - proceeding with total demand only")

## Train / Validation / Test Split

In [ ]:
n = len(ts_df)
ts_test     = ts_df.iloc[n-TEST_SIZE:].copy()
ts_val      = ts_df.iloc[n-TEST_SIZE-VAL_SIZE:n-TEST_SIZE].copy()
ts_train    = ts_df.iloc[:n-TEST_SIZE-VAL_SIZE].copy()
ts_trainval = ts_df.iloc[:n-TEST_SIZE].copy()

print(f"Train:  {len(ts_train):,} h ({ts_train['ds'].min().date()} -> {ts_train['ds'].max().date()})")
print(f"Val:    {len(ts_val):,} h  ({ts_val['ds'].min().date()} -> {ts_val['ds'].max().date()})")
print(f"Test:   {len(ts_test):,} h  ({ts_test['ds'].min().date()} -> {ts_test['ds'].max().date()})")
assert ts_train["ds"].max() < ts_val["ds"].min() < ts_test["ds"].min()
print("Clean split confirmed.")

## Preprocessing & Feature Engineering

In [ ]:
all_cols = list(ts_df.columns)
extra_feats = [c for c in all_cols if c not in ["ds","y","hour","dow"]]

def make_ride_features(df_in, extra):
    df = df_in.copy()
    for lag in [1, 2, 3, 24, 48, 168]:
        if lag < len(df): df[f"lag_{lag}"] = df["y"].shift(lag)
    df["roll_mean_24"]  = df["y"].shift(1).rolling(24).mean()
    df["roll_max_24"]   = df["y"].shift(1).rolling(24).max()
    df["roll_mean_168"] = df["y"].shift(1).rolling(168).mean() if len(df)>168 else 0
    df["hour_sin"]      = np.sin(2*np.pi*df["ds"].dt.hour/24)
    df["hour_cos"]      = np.cos(2*np.pi*df["ds"].dt.hour/24)
    df["dow_sin"]       = np.sin(2*np.pi*df["ds"].dt.dayofweek/7)
    df["dow_cos"]       = np.cos(2*np.pi*df["ds"].dt.dayofweek/7)
    df["is_weekend"]    = df["ds"].dt.dayofweek.isin([5,6]).astype(int)
    df["is_night"]      = df["ds"].dt.hour.isin([0,1,2,3,4,5]).astype(int)
    for c in extra: df[c] = df[c].fillna(df[c].median() if df[c].dtype.kind == "f" else 0)
    return df

ts_all  = pd.concat([ts_train, ts_val, ts_test]).reset_index(drop=True)
ts_feat = make_ride_features(ts_all, extra_feats)
lag_feats = [f"lag_{l}" for l in [1,2,3,24,48,168] if f"lag_{l}" in ts_feat.columns]
cal_feats = ["hour_sin","hour_cos","dow_sin","dow_cos","is_weekend","is_night",
             "roll_mean_24","roll_max_24","roll_mean_168"]
feat_cols = lag_feats + cal_feats + [c for c in extra_feats if c in ts_feat.columns]

split1 = len(ts_train); split2 = split1 + len(ts_val)
train_f = ts_feat.iloc[:split1].dropna().copy()
val_f   = ts_feat.iloc[split1:split2].dropna().copy()
test_f  = ts_feat.iloc[split2:].dropna().copy()
print(f"Features ({len(feat_cols)}): {feat_cols}")

## Baseline Approaches

In [ ]:
results = []
def evaluate(actual, pred, name):
    a,p = np.array(actual).flatten(), np.array(pred).flatten()
    n = min(len(a),len(p)); a,p = a[:n],p[:n]
    mae  = mean_absolute_error(a,p)
    rmse = np.sqrt(mean_squared_error(a,p))
    mape = np.mean(np.abs((a-p)/np.where(a>0,a,np.nan)))*100
    print(f"{name:<44s} MAE={mae:7.1f}  RMSE={rmse:7.1f}  MAPE={mape:.2f}%")
    results.append({"model":name,"MAE":mae,"RMSE":rmse,"MAPE":mape})

sn24  = np.tile(ts_trainval["y"].iloc[-SEASON_H:].values, len(ts_test)//SEASON_H+1)[:len(ts_test)]
sn168 = np.tile(ts_trainval["y"].iloc[-SEASON_W:].values, len(ts_test)//SEASON_W+1)[:len(ts_test)]
evaluate(ts_test["y"], np.full(len(ts_test), ts_trainval["y"].mean()), "Naive (global mean)")
evaluate(ts_test["y"], sn24, "Seasonal Naive (same hour yesterday)")
evaluate(ts_test["y"], sn168, "Seasonal Naive (same hour last week)")

## LazyPredict Benchmark

In [ ]:
X_tr = train_f[feat_cols]; y_tr = train_f["y"]
X_va = val_f[feat_cols];   y_va = val_f["y"]
try:
    lazy = LazyRegressor(verbose=0, ignore_warnings=True)
    lm, _ = lazy.fit(X_tr, X_va, y_tr, y_va)
    print(lm.head(12).to_string())
except Exception as e: print(f"LazyPredict: {e}")

## FLAML AutoML

In [ ]:
X_all = pd.concat([train_f, val_f])[feat_cols]
y_all = pd.concat([train_f, val_f])["y"]
X_te  = test_f[feat_cols]
flaml = AutoML()
flaml.fit(X_all, y_all, task="regression", metric="rmse",
          time_budget=FLAML_BUDGET, verbose=0, seed=RANDOM_STATE)
flaml_pred = np.maximum(flaml.predict(X_te), 0)
print(f"Best: {flaml.best_estimator}")
evaluate(test_f["y"], flaml_pred, f"FLAML ({flaml.best_estimator})")

## MLForecast - Time Series ML Library

In [ ]:
def to_mlf(df_in):
    d = df_in[["ds","y"]].copy()
    d.insert(0, "unique_id", "boston_rides")
    return d.dropna().reset_index(drop=True)

mlf_train = to_mlf(ts_trainval)

mlf = MLForecast(
    models={
        "LightGBM": lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05,
                                        num_leaves=64, random_state=RANDOM_STATE, verbose=-1),
        "XGBoost":  xgb.XGBRegressor(n_estimators=200, learning_rate=0.05,
                                       max_depth=6, random_state=RANDOM_STATE, verbosity=0),
    },
    freq="h",
    lags=[1, 2, 3, 24, 48, 168],
    lag_transforms={24: [("rolling_mean", 7)]},
    date_features=["hour","dayofweek"],
)

try:
    mlf.fit(mlf_train)
    mlf_pred = mlf.predict(HORIZON)
    print("MLForecast predictions:")
    print(mlf_pred)
    for col in ["LightGBM","XGBoost"]:
        if col in mlf_pred.columns:
            preds = np.maximum(mlf_pred[col].values, 0)
            evaluate(ts_test["y"].values[:len(preds)], preds, f"MLF-{col}")
except Exception as e:
    print(f"MLForecast error: {e}")
    mlf_pred = None

In [ ]:
# Plot forecast
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_test["ds"].iloc[:HORIZON], y=ts_test["y"].iloc[:HORIZON],
                          name="Actual", line=dict(color="black",width=2)))
if mlf_pred is not None:
    for col, clr in zip(["LightGBM","XGBoost"],["steelblue","darkorange"]):
        if col in mlf_pred.columns:
            fig.add_trace(go.Scatter(x=ts_test["ds"].iloc[:HORIZON],
                                      y=np.maximum(mlf_pred[col].values,0),
                                      name=f"MLF-{col}",
                                      line=dict(color=clr,dash="dash")))
fig.update_layout(title="Ride Demand - 24h MLForecast",
                  xaxis_title="Date/Time", yaxis_title="Rides per Hour",
                  template="plotly_white"); fig.show()

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print("ALL MODELS (ranked by RMSE)"); print(results_df.to_string(index=False))
top3 = results_df.head(3)
print("
TOP 3:"); print(top3.to_string(index=False))
fig = px.bar(results_df, x="model", y="RMSE",
             title="Ride Demand Model Comparison",
             color="RMSE", color_continuous_scale="RdYlGn_r")
fig.update_layout(xaxis_tickangle=-40, template="plotly_white"); fig.show()

## Error Analysis

In [ ]:
if len(test_f) > 0:
    actual = test_f["y"].values
    pred   = np.maximum(flaml.predict(test_f[feat_cols]), 0)
    errors = actual - pred

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].hist(errors, bins=25, color="steelblue", edgecolor="black", alpha=0.8)
    axes[0].axvline(0, color="red", ls="--"); axes[0].set_title("Residual Distribution")

    hours = test_f["ds"].dt.hour
    pd.Series(np.abs(errors)).groupby(hours.values).mean().plot(
        ax=axes[1], kind="bar", color="coral", alpha=0.8)
    axes[1].set_title("MAE by Hour of Day"); axes[1].set_xlabel("Hour")

    axes[2].scatter(actual, pred, s=4, alpha=0.3, color="steelblue")
    lo,hi = 0, max(actual.max(),pred.max())
    axes[2].plot([lo,hi],[lo,hi],"r--")
    axes[2].set_title("Actual vs Predicted")
    axes[2].set_xlabel("Actual (rides/hr)"); axes[2].set_ylabel("Predicted")
    plt.tight_layout(); plt.show()

## Interpretation & Insights

1. **lag_168 (same hour last week)** is critical - ride-hailing demand has very consistent weekly patterns (Friday 11PM always busy, Monday 6AM always slow)
2. **LightGBM with weather covariates** outperforms pure lag-based models by capturing rain/cold suppression effects
3. **Night hours (1-4AM) are hardest to forecast** - stochastic late-night social activity; but also lowest impact errors in absolute terms
4. **Surge multiplier as a feature is problematic**: surge is determined simultaneously with demand (endogenous); including it leaks future information; use lagged surge instead
5. **Weekend demand profile** is flatter and more uniformly distributed vs. weekday; the model needs to learn both profiles distinctly

## Limitations

1. **Only 2 months of data** (Nov-Dec 2018) - seasonal variation from summer cannot be captured
2. **Boston-specific**: demand patterns, weather effects, and event calendars are locality-specific; models do not generalise to other cities without retraining
3. **No event calendar**: concerts, Red Sox/Celtics games create unmissable demand spikes not captured by weather or lag features
4. **Surge endogeneity**: using current-hour surge as a feature leaks demand signal; only lagged surge is valid
5. **Missing GPS granularity**: aggregating city-wide hides spatial hot-spots (airport vs downtown vs university)

## How to Improve This Project

1. **Event calendar integration**: merge a Boston events calendar (Ticketmaster API) to add `event_capacity_evening` feature
2. **Per-neighbourhood forecast**: disaggregate demand by `source` neighbourhood; run MLForecast on a panel of 20+ location time series
3. **Lag surge feature**: add `avg_surge_24_lag` (surge multiplier same hour yesterday) as a demand predictor
4. **Probabilistic forecast**: use LightGBM quantile regression (alpha=0.1, 0.5, 0.9) to produce driver incentive dispatch scenarios
5. **Longer history**: combine with the original TLC data from NYC for a 5-year multi-city training corpus

## Production Considerations

1. **Real-time dispatch**: feed 15-min ahead forecast to the driver positioning model (geohash grid level)
2. **Surge pre-activation**: if 30-min forecast exceeds expected supply by >20%, pre-activate 1.1x surge to attract driver supply before demand hits
3. **Model refresh**: daily full retrain on rolling 90-day window; hourly bias correction using recent 3 days of actual vs forecast
4. **City-specific instances**: deploy one MLForecast model per city (Boston, NYC, Chicago) rather than one global model
5. **Confidence intervals**: expose P10/P90 bands to the driver incentive system for risk-aware supply positioning

## Common Mistakes to Avoid

1. **Not separating Uber and Lyft before aggregation**: if the goal is per-platform demand, mixing them produces a spurious composite series
2. **Using raw Unix timestamps without floor rounding**: partial-hour boundaries create uneven bucket sizes and false demand spikes
3. **Including current surge multiplier as a feature**: it is simultaneous with demand (endogenous), causing data leakage
4. **Not handling night-time zero demand**: some 3-4AM hours have near-zero rides; MAPE gives huge errors at these hours - use RMSE or WMAPE
5. **Ignoring weather timezone**: weather data may be in UTC while demand timestamps are local; always align to the same timezone before merging

## Mini Challenge / Exercises

1. **Disaggregate by cab type**: separately forecast Uber vs Lyft hourly demand. Do they peak at the same hours?
2. **Per-neighbourhood demand**: pick the top 3 busiest source neighbourhoods and run a panel MLForecast. Compare accuracy across locations.
3. **Surge prediction**: instead of forecasting demand counts, predict the `surge_multiplier` for the next hour. What features are most important?
4. **Weather threshold**: create binary features `is_raining` (rain > 0.1 in), `is_very_cold` (temp < 32F), `is_night_out` (Fri/Sat 10PM-4AM). Report RMSE change.
5. **Seasonality by time-of-day**: compute MAPE separately for peak (7-9AM, 5-7PM), social (10PM-2AM), and overnight (3-6AM) hours. Which period is forecast best?

## Final Summary & Key Takeaways

### What We Did
- Downloaded individual Uber/Lyft trip records; aggregated to hourly ride counts
- Merged weather covariates (temp, rain, humidity, wind, clouds) and surge multiplier by hour
- Analysed weekday vs weekend demand profiles and weather correlations
- Compared Uber vs Lyft demand volumes
- Built ride-specific features: lag_24, lag_168, is_night, weather exogenous regressors
- Benchmarked with LazyPredict; ran FLAML AutoML
- Fitted MLForecast with LightGBM and XGBoost
- Selected Top 3 by RMSE; analysed errors by hour-of-day

### Key Takeaways
1. **lag_168 (same hour last week) is the strongest predictor** for ride demand - weekly calendar is more stable than daily patterns for ride-hailing
2. **Weather matters but is secondary** - rain boosts demand slightly, extreme cold reduces it; the temporal pattern dominates
3. **Surge multiplier should be lagged** when used as a feature - current-hour surge is endogenous; only past surge is valid as an input
4. **Night-time hours (2-5AM) are inherently unpredictable** - stochastic late-night social activity; park these errors separately in your RMSE reports
5. **MLForecast with lag features is the practical choice** - scaling to 100+ city time series in parallel with minimal code changes is its core advantage

---
*Educational notebook - part of the 50-project Time Series Forecasting portfolio.*